# Comparación controlada de 822 transformaciones JPG frente a PNG

Este cuaderno verifica primero que las 822 imágenes originales balanceadas sean idénticas en píxeles a sus equivalentes del dataset completo. Después calcula, en una sola ejecución y con el mismo modelo, dispositivo y preprocesamiento, las métricas original–JPG y original–PNG. No modifica ni regenera las imágenes existentes.

In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [2]:
from pathlib import Path

BASE_DIR = Path('/content/gdrive/MyDrive/2026-0')
COMPLETE_ROOT = BASE_DIR / 'DatasetOriginalCompleto'
BALANCED_ROOT = BASE_DIR / 'DatasetOriginal-137-Por-Clase'
JPG_ROOT = BASE_DIR / 'TrashNet_transformaciones'
PNG_ROOT = BASE_DIR / 'TrashNet_transformaciones_completo_png'
RESULTS_ROOT = BASE_DIR / 'Resultados_comparacion_controlada_822_JPG_PNG'
CONDITION_ROOT = RESULTS_ROOT / 'condiciones'
FINAL_CSV = RESULTS_ROOT / 'comparacion_controlada_822_JPG_PNG.csv'
AUDIT_CSV = RESULTS_ROOT / 'auditoria_originales_822.csv'
METADATA_JSON = RESULTS_ROOT / 'entorno_comparacion_controlada.json'
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
CONDITION_ROOT.mkdir(parents=True, exist_ok=True)

CLASSES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
CONDITIONS = [
    ('brightness', 'low', 1.2), ('brightness', 'medium', 1.4), ('brightness', 'high', 1.6),
    ('contrast', 'low', 1.2), ('contrast', 'medium', 1.4), ('contrast', 'high', 1.6),
    ('rotation', 'low', 10), ('rotation', 'medium', 30), ('rotation', 'high', 60),
    ('zoom', 'low', 1.10), ('zoom', 'medium', 1.25), ('zoom', 'high', 1.50),
]
VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp'}
BATCH_SIZE = 8
SEED = 2026

for path in [COMPLETE_ROOT, BALANCED_ROOT, JPG_ROOT, PNG_ROOT]:
    if not path.exists():
        raise FileNotFoundError(f'No existe la carpeta requerida: {path}')

print('Original completo:', COMPLETE_ROOT)
print('Original balanceado:', BALANCED_ROOT)
print('Transformaciones JPG:', JPG_ROOT)
print('Transformaciones PNG:', PNG_ROOT)
print('Resultados:', RESULTS_ROOT)

Original completo: /content/gdrive/MyDrive/2026-0/DatasetOriginalCompleto
Original balanceado: /content/gdrive/MyDrive/2026-0/DatasetOriginal-137-Por-Clase
Transformaciones JPG: /content/gdrive/MyDrive/2026-0/TrashNet_transformaciones
Transformaciones PNG: /content/gdrive/MyDrive/2026-0/TrashNet_transformaciones_completo_png
Resultados: /content/gdrive/MyDrive/2026-0/Resultados_comparacion_controlada_822_JPG_PNG


## 1. Auditoría de identidad de las 822 imágenes originales

In [3]:
import hashlib
import numpy as np
import pandas as pd
from PIL import Image

def list_images(folder):
    return sorted([p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS], key=lambda p: p.name.lower())

def sha256(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

audit_rows = []
balanced_items = []
for class_name in CLASSES:
    balanced_paths = list_images(BALANCED_ROOT / class_name)
    if len(balanced_paths) != 137:
        raise ValueError(f'{class_name}: se esperaban 137 imágenes balanceadas y se encontraron {len(balanced_paths)}')
    for balanced_path in balanced_paths:
        complete_path = COMPLETE_ROOT / class_name / balanced_path.name
        if not complete_path.exists():
            raise FileNotFoundError(f'No existe la contraparte completa: {complete_path}')
        hash_equal = sha256(balanced_path) == sha256(complete_path)
        with Image.open(balanced_path) as a, Image.open(complete_path) as b:
            arr_a = np.asarray(a.convert('RGB'))
            arr_b = np.asarray(b.convert('RGB'))
        pixel_equal = arr_a.shape == arr_b.shape and np.array_equal(arr_a, arr_b)
        audit_rows.append({
            'class': class_name, 'filename': balanced_path.name,
            'sha256_equal': hash_equal, 'pixel_equal': pixel_equal,
            'balanced_shape': str(arr_a.shape), 'complete_shape': str(arr_b.shape)
        })
        balanced_items.append((class_name, balanced_path.name, complete_path))

audit = pd.DataFrame(audit_rows)
audit.to_csv(AUDIT_CSV, index=False)
print('Imágenes auditadas:', len(audit))
print('Idénticas byte por byte:', int(audit.sha256_equal.sum()))
print('Idénticas en píxeles:', int(audit.pixel_equal.sum()))
print('Auditoría guardada en:', AUDIT_CSV)
if len(audit) != 822 or not audit.pixel_equal.all():
    display(audit[~audit.pixel_equal].head(20))
    raise ValueError('La comparación se detuvo: existen originales balanceados que no coinciden en píxeles con el dataset completo.')

Imágenes auditadas: 822
Idénticas byte por byte: 822
Idénticas en píxeles: 822
Auditoría guardada en: /content/gdrive/MyDrive/2026-0/Resultados_comparacion_controlada_822_JPG_PNG/auditoria_originales_822.csv


## 2. Verificación de las transformaciones JPG y PNG

In [4]:
missing = []
for class_name, filename, _ in balanced_items:
    stem = Path(filename).stem
    for transformation, level, _ in CONDITIONS:
        jpg_path = JPG_ROOT / f'{transformation}_{level}' / class_name / filename
        png_path = PNG_ROOT / f'{transformation}_{level}' / class_name / f'{stem}.png'
        if not jpg_path.exists(): missing.append(str(jpg_path))
        if not png_path.exists(): missing.append(str(png_path))
print('Archivos esperados:', 822 * 12 * 2)
print('Archivos faltantes:', len(missing))
if missing:
    print(*missing[:20], sep='\n')
    raise FileNotFoundError('Faltan transformaciones; revisa las rutas mostradas arriba.')

Archivos esperados: 19728
Archivos faltantes: 0


## 3. VGG-16 y preprocesamiento común

In [5]:
import json, platform, random, sys
import torch
import torch.nn.functional as TF
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import vgg16, VGG16_Weights
from tqdm.auto import tqdm

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
weights = VGG16_Weights.IMAGENET1K_V1
model = vgg16(weights=weights).to(device).eval()
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
selected_layers = {'Block1': 3, 'Block2': 8, 'Block3': 15, 'Block4': 22, 'Block5': 29}
activations = {}
hooks = []
def save_activation(name):
    def hook(_, __, output):
        activations[name] = output.detach()
    return hook
for name, index in selected_layers.items():
    hooks.append(model.features[index].register_forward_hook(save_activation(name)))
print('Dispositivo:', device)
print('Preprocesamiento:', preprocess)

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:07<00:00, 73.8MB/s]


Dispositivo: cuda
Preprocesamiento: Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


## 4. Comparación pareada por condición

In [6]:
class TripletDataset(Dataset):
    def __init__(self, transformation, level):
        self.transformation = transformation
        self.level = level
    def __len__(self): return len(balanced_items)
    def __getitem__(self, index):
        class_name, filename, original_path = balanced_items[index]
        stem = Path(filename).stem
        jpg_path = JPG_ROOT / f'{self.transformation}_{self.level}' / class_name / filename
        png_path = PNG_ROOT / f'{self.transformation}_{self.level}' / class_name / f'{stem}.png'
        with Image.open(original_path) as im: original = preprocess(im.convert('RGB'))
        with Image.open(jpg_path) as im: jpg = preprocess(im.convert('RGB'))
        with Image.open(png_path) as im: png = preprocess(im.convert('RGB'))
        return original, jpg, png, class_name, stem, filename

def metrics(a, b):
    euclidean = torch.linalg.vector_norm(a - b, ord=2, dim=1)
    relative = euclidean / (torch.linalg.vector_norm(a, ord=2, dim=1) + 1e-8)
    cosine = TF.cosine_similarity(a, b, dim=1)
    return euclidean.cpu().numpy(), relative.cpu().numpy(), cosine.cpu().numpy()

for transformation, level, value in CONDITIONS:
    condition_path = CONDITION_ROOT / f'{transformation}_{level}.csv'
    expected_rows = 822 * 5
    if condition_path.exists():
        prior = pd.read_csv(condition_path)
        if len(prior) == expected_rows and not prior.duplicated(['class', 'image_id', 'block']).any():
            print('Condición ya completa, se conserva:', transformation, level)
            continue
        print('Condición incompleta; se recalcula:', transformation, level)
    loader = DataLoader(TripletDataset(transformation, level), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    rows = []
    for original, jpg, png, class_names, stems, filenames in tqdm(loader, desc=f'{transformation}_{level}'):
        batch_n = original.shape[0]
        x = torch.cat([original, jpg, png], dim=0).to(device)
        activations.clear()
        with torch.inference_mode():
            _ = model.features(x)
        for block, feature_map in activations.items():
            embedding = feature_map.mean(dim=(2, 3))
            emb_original = embedding[:batch_n]
            emb_jpg = embedding[batch_n:2*batch_n]
            emb_png = embedding[2*batch_n:]
            e_jpg, r_jpg, c_jpg = metrics(emb_original, emb_jpg)
            e_png, r_png, c_png = metrics(emb_original, emb_png)
            e_direct, r_direct, c_direct = metrics(emb_jpg, emb_png)
            for i in range(batch_n):
                rows.append({
                    'image_id': stems[i], 'original_filename': filenames[i], 'class': class_names[i],
                    'transformation': transformation, 'level': level, 'value': value, 'block': block,
                    'euclidean_jpg': float(e_jpg[i]), 'euclidean_png': float(e_png[i]),
                    'delta_euclidean_png_minus_jpg': float(e_png[i] - e_jpg[i]),
                    'relative_euclidean_jpg': float(r_jpg[i]), 'relative_euclidean_png': float(r_png[i]),
                    'delta_relative_png_minus_jpg': float(r_png[i] - r_jpg[i]),
                    'cosine_jpg': float(c_jpg[i]), 'cosine_png': float(c_png[i]),
                    'delta_cosine_png_minus_jpg': float(c_png[i] - c_jpg[i]),
                    'euclidean_jpg_vs_png': float(e_direct[i]),
                    'relative_euclidean_jpg_vs_png': float(r_direct[i]),
                    'cosine_jpg_vs_png': float(c_direct[i]),
                })
    condition_df = pd.DataFrame(rows)
    if len(condition_df) != expected_rows or condition_df.duplicated(['class', 'image_id', 'block']).any():
        raise ValueError(f'La condición {transformation}_{level} no superó la auditoría.')
    condition_df.to_csv(condition_path, index=False)
    print('Guardado:', condition_path, '| filas:', len(condition_df))

brightness_low:   0%|          | 0/103 [00:00<?, ?it/s]

Guardado: /content/gdrive/MyDrive/2026-0/Resultados_comparacion_controlada_822_JPG_PNG/condiciones/brightness_low.csv | filas: 4110


brightness_medium:   0%|          | 0/103 [00:00<?, ?it/s]

Guardado: /content/gdrive/MyDrive/2026-0/Resultados_comparacion_controlada_822_JPG_PNG/condiciones/brightness_medium.csv | filas: 4110


brightness_high:   0%|          | 0/103 [00:00<?, ?it/s]

...

Guardado: /content/gdrive/MyDrive/2026-0/Resultados_comparacion_controlada_822_JPG_PNG/condiciones/zoom_low.csv | filas: 4110


zoom_medium:   0%|          | 0/103 [00:00<?, ?it/s]

Guardado: /content/gdrive/MyDrive/2026-0/Resultados_comparacion_controlada_822_JPG_PNG/condiciones/zoom_medium.csv | filas: 4110


zoom_high:   0%|          | 0/103 [00:00<?, ?it/s]

Guardado: /content/gdrive/MyDrive/2026-0/Resultados_comparacion_controlada_822_JPG_PNG/condiciones/zoom_high.csv | filas: 4110


## 5. Consolidación y auditoría final

In [7]:
frames = []
for transformation, level, _ in CONDITIONS:
    path = CONDITION_ROOT / f'{transformation}_{level}.csv'
    if not path.exists(): raise FileNotFoundError(f'Falta {path}')
    frames.append(pd.read_csv(path))
final = pd.concat(frames, ignore_index=True)
key = ['class', 'image_id', 'transformation', 'level', 'block']
numeric = [c for c in final.columns if c.startswith(('euclidean', 'relative', 'cosine', 'delta'))]
if len(final) != 49320: raise ValueError(f'Se esperaban 49320 filas y se obtuvieron {len(final)}')
if final.duplicated(key).any(): raise ValueError('Existen filas duplicadas.')
if final.isna().any().any(): raise ValueError('Existen valores ausentes.')
if not np.isfinite(final[numeric].to_numpy()).all(): raise ValueError('Existen valores no finitos.')
final.to_csv(FINAL_CSV, index=False)
metadata = {
    'seed': SEED, 'python': platform.python_version(), 'torch': torch.__version__,
    'torchvision': torchvision.__version__, 'device': str(device),
    'weights': str(weights), 'preprocessing': str(preprocess),
    'layer_indices': selected_layers, 'images': 822, 'conditions': 12, 'rows': len(final)
}
with METADATA_JSON.open('w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
for hook in hooks: hook.remove()
print('COMPARACIÓN COMPLETA Y VALIDADA')
print('CSV final:', FINAL_CSV)
print('Filas:', len(final))
display(final.head())

COMPARACIÓN COMPLETA Y VALIDADA
CSV final: /content/gdrive/MyDrive/2026-0/Resultados_comparacion_controlada_822_JPG_PNG/comparacion_controlada_822_JPG_PNG.csv
Filas: 49320


,image_id,original_filename,class,transformation,level,value,block,euclidean_jpg,euclidean_png,delta_euclidean_png_minus_jpg,relative_euclidean_jpg,relative_euclidean_png,delta_relative_png_minus_jpg,cosine_jpg,cosine_png,delta_cosine_png_minus_jpg,euclidean_jpg_vs_png,relative_euclidean_jpg_vs_png,cosine_jpg_vs_png
0,cardboard1,cardboard1.jpg,cardboard,brightness,low,1.2,Block1,1.427624,1.465172,0.037547,0.356562,0.365939,0.009378,0.968341,0.968212,-0.000129,0.073662,0.015044,0.999939
1,cardboard10,cardboard10.jpg,cardboard,brightness,low,1.2,Block1,1.591756,1.624539,0.032784,0.417290,0.425884,0.008594,0.956658,0.956918,0.000260,0.078352,0.016360,0.999920
2,cardboard100,cardboard100.jpg,cardboard,brightness,low,1.2,Block1,1.771416,1.777753,0.006337,0.563482,0.565497,0.002016,0.926717,0.927332,0.000616,0.028099,0.006638,0.999984
3,cardboard101,cardboard101.jpg,cardboard,brightness,low,1.2,Block1,0.857444,0.869547,0.012103,0.226747,0.229947,0.003201,0.978966,0.978827,-0.000138,0.056658,0.013900,0.999918
4,cardboard102,cardboard102.jpg,cardboard,brightness,low,1.2,Block1,1.350481,1.356945,0.006464,0.435833,0.437919,0.002086,0.944125,0.944947,0.000822,0.053117,0.013945,0.999918
